# Giai đoạn 0 · Đánh giá policy tham chiếu trên 30 kịch bản

Chạy policy của notebook [Smart Buildings Probe DS K07](https://www.kaggle.com/code/malyshevdanil/smart-buildings-probe-ds-k07)
(public LB 0,01582, bản 21/10/2026) trên đủ 30 kịch bản train, mỗi kịch bản 8.640 bước.

| Ra | Ý nghĩa |
|---|---|
| Điểm từng kịch bản | Tách thành tiện nghi · năng lượng · carbon |
| Phần đóng góp | Kịch bản nào chiếm nhiều điểm nhất |
| Nhóm null | Mức nhiễu: thay đổi nhỏ hơn mức này là vô nghĩa |
| Thời gian | So với giới hạn 10× bang-bang |

**Chạy trên Kaggle:** gắn dữ liệu cuộc thi, chọn CPU, rồi *Save Version → Save & Run All*.
Ước tính 2–4 giờ với 4 nhân. Kết quả nằm trong `eval_ref/`. Nếu phiên bị ngắt, chạy lại sẽ bỏ qua kịch bản đã xong.

**Chạy máy khác:** đặt `SMART_BUILDINGS_DATA_ROOT` trỏ tới bản clone starter kit (Python 3.12).

In [ ]:
import os, sys, json, time
from pathlib import Path

# One thread per simulator process; parallelism comes from running scenarios side by side.
for k in ("NUMBA_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[k] = "1"

N_STEPS = int(os.getenv("SB_STEPS", 8640))      # 8640 = full 30-day episode. Shorter runs are smoke tests only.
WORKERS = int(os.getenv("SB_WORKERS", os.cpu_count() or 1))
ONLY = [s for s in os.getenv("SB_ONLY", "").split(",") if s]   # e.g. "building_train_8/phoenix_4"
RUN_NULL = os.getenv("SB_NULL", "1") == "1"     # null group: noise level
NULL_TOP = int(os.getenv("SB_NULL_TOP", 5))     # on the scenarios with the largest share of the score
NULL_K = int(os.getenv("SB_NULL_K", 3))         # nudged reruns per scenario
RUN_TIMING = os.getenv("SB_TIMING", "1") == "1"
TIMING_STEPS = int(os.getenv("SB_TIMING_STEPS", 576))

candidates = [os.getenv("SMART_BUILDINGS_DATA_ROOT", ""),
              "/kaggle/input/competitions/neurips-2026-smart-buildings",
              "/kaggle/input/neurips-2026-smart-buildings",
              "../starter-kit", "starter-kit"]
DATA_ROOT = next((Path(p).resolve() for p in candidates if p and (Path(p) / "smart_buildings").exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Attach the competition data, or set SMART_BUILDINGS_DATA_ROOT to a starter-kit checkout")

WORK = Path.cwd().resolve()
OUT = WORK / "eval_ref"
OUT.mkdir(parents=True, exist_ok=True)
os.environ["SB_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SB_WORK"] = str(WORK)
for p in (str(WORK), str(DATA_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print(f"data:    {DATA_ROOT}\nout:     {OUT}\nsteps:   {N_STEPS}\nworkers: {WORKERS}")

## Policy

Code nguyên văn từ notebook tham chiếu (Apache-2.0). Phần thêm ở cuối file: `RefPolicy` mang đúng tham số của `SubmissionPolicy`,
`make_policy` tạo biến thể null (lệch áp suất ~0,01 Pa) và bang-bang để đo thời gian.

In [ ]:
%%writefile ref_policy.py
"""Reference controller (public LB 0.01582) for offline evaluation.

Code copied verbatim from the Kaggle notebook "Smart Buildings Probe DS K07" by
Malyshev Danil (Apache-2.0), version run on 2026-10-21:
https://www.kaggle.com/code/malyshevdanil/smart-buildings-probe-ds-k07
It builds on the organizers' PIDWaterResetPolicy (Apache-2.0).

Only additions, below the copied code: `RefPolicy` carries the submitted
constants without the Kaggle inference-server base class, and `make_policy`
adds nudged "null" variants and the bang-bang baseline for the noise and
runtime checks.
"""
import numpy as np
from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy


class BaselinePolicy(PIDWaterResetPolicy):
    USE_SENSOR_BAND = True
    # Worst-zone aggregation increased loss in both full-length pilot scenarios.
    USE_WORST_ZONE = False

    @property
    def name(self):
        return "transfer"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        stats = self.observation_normalization_metadata.get(
            "outside_air_temperature_sensor", {}
        )
        self.outside_mean = float(stats.get("sample_mean", 0.0))
        self.outside_std = float(stats.get("sample_std", 1.0))

    def _recirculation(self, obs, ctrl, u, low, high):
        # The parent's economizer expects kelvin; the observation is standardized.
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        decoded = obs.copy()
        decoded[idx] = obs[idx] * self.outside_std + self.outside_mean
        return super()._recirculation(decoded, ctrl, u, low, high)

    def get_action(self, obs):
        action = super().get_action(obs)
        # Enforce the gateway's dtype, shape and bounds contract.
        for key, dtype in (("continuous_action", np.float32),
                           ("discrete_action", np.int32)):
            space = self.action_spec[key]
            action[key] = np.clip(action[key], space.low, space.high).astype(dtype)
        return action


class CoolingHoldPolicy(BaselinePolicy):
    HOLD_STEPS = 288
    # Outside work hours use (eco low, comfort high): no overnight drift up to
    # the eco ceiling in hot climates, full heating setback kept in cold ones.
    # Idea from the public "Neurips 2026 Smart Buildings Smartpolicy" notebook.
    NIGHT_COOL_CEILING = False

    @property
    def name(self):
        return f"cooling_hold_{self.HOLD_STEPS}"

    def _base_action(self, obs):
        if not self.NIGHT_COOL_CEILING:
            return super().get_action(obs)
        hour, day = self._get_hour_and_day(obs)
        if day < 5 and self.morning_start_hour <= hour < self.evening_start_hour:
            return super().get_action(obs)
        saved = self.eco_temp_window
        self.eco_temp_window = (saved[0], self.comfort_temp_window[1])
        try:
            return super().get_action(obs)
        finally:
            self.eco_temp_window = saved

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._on_steps = [0] * len(self.ahu_controllers)
        self._last_on = [None] * len(self.ahu_controllers)

    def get_action(self, obs):
        action = self._base_action(obs)
        cont, disc = action["continuous_action"], action["discrete_action"]
        for i, ctrl in enumerate(self.ahu_controllers):
            if not ctrl["zone_obs_indices"]:
                continue
            run = ctrl["run_discrete_idx"]
            # Supply temperature first: its sign tells cooling from heating.
            idx = [j for j in (ctrl["temp_idx"], ctrl["pressure_idx"], ctrl.get("recirc_idx"))
                   if j is not None]
            if (not disc[run] and self._on_steps[i] > 0
                    and self._on_steps[i] < self.HOLD_STEPS and self._last_on[i][0] < 0):
                disc[run] = 1
                cont[idx] = self._last_on[i]
            if disc[run]:
                self._on_steps[i] += 1
                self._last_on[i] = cont[idx].copy()
            else:
                self._on_steps[i] = 0
        return action


class ZonePIPolicy(CoolingHoldPolicy):
    """v5 controller with per-zone PI control to the band centre in work hours.

    A soft proportional zone loop stops the 2-4 K oscillation but leaves zones
    sitting steadily outside the band (no integral action). Here each zone runs
    a PI loop on (temperature - band centre) from one hour before occupancy to
    the end of the work day. One signal u in [-1, 1] drives both actuators:
    the damper opens with |u| (air is needed to cool and to carry reheat) and
    the reheat valve opens with -u. AHUs are kept running in those hours so the
    zones have air to work with. Outside them, v5 logic is unchanged.
    """
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = True
    KP = 0.2
    KI = 0.02
    PRESTART_H = 1
    # Hours before the end of the work day at which the PI loops stop (coast).
    EVENING_STOP_H = 0
    FORCED_SAT = -0.7       # normalized supply temp for a forced-on AHU (~15 C)
    FORCED_PRESSURE = -0.6  # normalized static pressure for a forced-on AHU
    KEY = "zone_pi"

    @property
    def name(self):
        return self.KEY

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._zone_integral = {}
        self._zone_u = {}
        self._pi_active = False

    def _pi_hours(self, obs):
        hour, day = self._get_hour_and_day(obs)
        return (day < 5 and self.morning_start_hour - self.PRESTART_H <= hour
                < self.evening_start_hour - self.EVENING_STOP_H)

    def _command_zones(self, continuous, zones, obs, low, high):
        if not self._pi_active:
            for vav_id, _, _ in zones:
                self._zone_integral[vav_id] = 0.0
            return super()._command_zones(continuous, zones, obs, low, high)
        lo_c, hi_c = self.comfort_temp_window
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            lo, hi = self._zone_band(obs, vav_id, lo_c, hi_c)
            # Before occupancy the sensor band is still the eco band; aim at comfort.
            if hi - lo > (hi_c - lo_c) + 1e-6:
                lo, hi = lo_c, hi_c
            self._cur_vav = vav_id
            err = self._denormalize_zone_temp(obs[obs_idx]) - self._zone_target(obs, lo, hi)
            integral = self._zone_integral.get(vav_id, 0.0)
            u_raw = self.KP * err + self.KI * integral
            # Conditional integration: stop winding up while saturated.
            integral *= 1.0 - self.INTEGRAL_LEAK
            if abs(u_raw) < 1.0 or np.sign(err) != np.sign(u_raw):
                limit = self.INTEGRAL_LIMIT_FRAC / self.KI
                integral = float(np.clip(integral + err, -limit, limit))
            self._zone_integral[vav_id] = integral
            u = float(np.clip(self.KP * err + self.KI * integral, -1.0, 1.0))
            if u < 0:
                u *= self.HEAT_U_SCALE
            self._max_heat = max(self._max_heat, -u)
            self._zone_u[vav_id] = u
            gain = self.HEAT_DAMPER_GAIN if u < 0 else 1.0
            damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * abs(u) * gain
            reheat = max(0.0, -u)
            if self.REHEAT_FULL_AT is not None and u < 0:
                # Hottest possible discharge for the air that is moved; the
                # damper alone sets how much heat reaches the zone.
                reheat = min(1.0, -u / self.REHEAT_FULL_AT)
            split = self.HEAT_SEQUENCE_SPLIT
            if split is not None and u < 0:
                # Reheat first at minimum air; add air only once reheat is full.
                reheat = min(1.0, -u / split)
                damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * max(0.0, (-u - split) / (1.0 - split))
            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                continuous[reheat_idx] = 2.0 * reheat - 1.0

    MIN_DAMPER = 0.1
    # Damper opening per unit of heating demand (reheated air costs gas).
    HEAT_DAMPER_GAIN = 1.0
    # Anti-windup: integral term limited to this share of full output, and a
    # per-step leak towards zero (both neutral by default).
    INTEGRAL_LIMIT_FRAC = 1.0
    INTEGRAL_LEAK = 0.0
    # Heating sequence: None = damper and reheat both follow |u|; a number s =
    # reheat opens fully by |u| = s at minimum air, then the damper opens.
    HEAT_SEQUENCE_SPLIT = None
    # Reheat valve fully open once heating demand reaches this |u| (None =
    # valve follows |u|): each unit of cold mixed air arrives as hot as allowed.
    REHEAT_FULL_AT = None
    # Scale of the PI output when heating (softer loop for hot-discharge heating).
    HEAT_U_SCALE = 1.0

    def _zone_target(self, obs, lo, hi):
        """Temperature each zone's PI loop aims at; the band centre by default."""
        return 0.5 * (lo + hi)

    def _base_action(self, obs):
        self._pi_active = self._pi_hours(obs)
        self._max_heat = 0.0
        self._forced = []
        self._zone_u = {}
        action = super()._base_action(obs)
        if self._pi_active:
            cont, disc = action["continuous_action"], action["discrete_action"]
            # Zone reheat needs hot water even when no AHU loop asks for heat.
            if self.water_temp_idx is not None and self._max_heat > 0:
                cont[self.water_temp_idx] = max(cont[self.water_temp_idx],
                                                np.float32(2.0 * self._max_heat - 1.0))
                if self.water_pressure_idx is not None:
                    cont[self.water_pressure_idx] = 1.0
            for i, ctrl in enumerate(self.ahu_controllers):
                if ctrl["zone_obs_indices"] and not disc[ctrl["run_discrete_idx"]]:
                    self._forced.append(i)
                    disc[ctrl["run_discrete_idx"]] = 1
                    cont[ctrl["temp_idx"]] = self.FORCED_SAT
                    if ctrl["pressure_idx"] is not None:
                        cont[ctrl["pressure_idx"]] = self.FORCED_PRESSURE
            if 0 <= self.water_discrete_idx < self.n_discrete:
                disc[self.water_discrete_idx] = 1
        return action

    def get_action(self, obs):
        action = super().get_action(obs)
        # A forced-on unit is not a cooling run; do not let the hold extend it.
        for i in self._forced:
            if self._last_on[i] is not None:
                self._last_on[i][0] = 0.0
        return action


def _outside_kelvin(policy, value):
    """AHU outside-air sensors arrive in raw kelvin; decode only if normalized."""
    value = float(value)
    return value if value > 150.0 else value * policy.outside_std + policy.outside_mean


class ZonePIEconFixPolicy(ZonePIPolicy):
    """pe_lowp with a working economizer (raw-kelvin outside air temperature).

    TransferPolicy decoded the AHU outside-air sensor as if normalized, but it
    arrives in kelvin, so the economizer saw ~3000 K outside and never opened.
    """
    KP = 0.10
    KI = 0.005
    FORCED_PRESSURE = -1.0
    WARM_SAT = None
    WARM_BELOW_K = 283.15
    KEY = "econ_fix"

    def _recirculation(self, obs, ctrl, u, low, high):
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        fixed = obs.copy()
        fixed[idx] = _outside_kelvin(self, obs[idx])
        # Skip TransferPolicy's decoding; call the organizer economizer directly.
        return super(BaselinePolicy, self)._recirculation(fixed, ctrl, u, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if self.WARM_SAT is None:
            return action
        cont = action["continuous_action"]
        for i in self._forced:
            ctrl = self.ahu_controllers[i]
            idx = ctrl.get("oat_idx")
            if idx is not None and _outside_kelvin(self, obs[idx]) < self.WARM_BELOW_K:
                cont[ctrl["temp_idx"]] = self.WARM_SAT
        return action


class DemandSatPolicy(ZonePIEconFixPolicy):
    """Supply-air temperature follows the zones' own PI demand in work hours.

    Forced-on AHUs used a fixed ~15 C supply. In cool weather that chills an
    already cool mix, then zone reheat burns gas to warm it back. Here each
    running AHU looks at the mean PI output of its zones (positive = cooling):
    with net heating demand the supply setpoint goes to maximum (no chiller);
    with cooling demand it ramps down to ~15 C, reached at mean demand RAMP
    (RAMP 0 = switch at zero demand).
    """
    RAMP = 0.0
    COLD_SAT = -0.7
    KEY = "dsat_bin"

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if not self._pi_active:
            return action
        cont, disc = action["continuous_action"], action["discrete_action"]
        for ctrl in self.ahu_controllers:
            us = [self._zone_u[v] for v, _, _ in ctrl["zones"] if v in self._zone_u]
            if not us or not disc[ctrl["run_discrete_idx"]]:
                continue
            d = float(np.mean(us))
            frac = (float(np.clip(d / self.RAMP, 0.0, 1.0)) if self.RAMP > 0
                    else (1.0 if d > 0 else 0.0))
            cont[ctrl["temp_idx"]] = np.float32(1.0 + frac * (self.COLD_SAT - 1.0))
        return action


class TunedPIPolicy(ZonePIEconFixPolicy):
    """econ_fix_warmcold plus switchable energy measures, all off by default.

    NIGHT_HEAT_DROP_K: lower the heating setpoint outside PI hours.
    WATER_SCALE: scale the hot-water temperature command (0..1 of range).
    TARGET_SHIFT_K: aim zones this far from the band centre towards the side
        the weather pushes them to (cheaper to hold).
    OCC_GATE: after the arrival hour, run PI hours only while the building
        reports occupants (skips holidays and empty days).
    """
    WARM_SAT = -0.4
    NIGHT_HEAT_DROP_K = 0.0
    WATER_SCALE = 1.0
    TARGET_SHIFT_K = 0.0
    OCC_GATE = False
    # Shift the target towards the low edge only in heating weather.
    COLD_SHIFT_ONLY = False
    # In PI hours, run the hot-water loop at full temperature whenever any
    # zone calls for heat (more heat per unit of cold mixed air).
    WATER_MAX_PI = False
    KEY = "tuned"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        names = list(self.field_names)
        self._occ_idx = names.index("num_occupants") if "num_occupants" in names else None
        self._oat_any = next((c["oat_idx"] for c in self.ahu_controllers
                              if c.get("oat_idx") is not None), None)

    def _pi_hours(self, obs):
        active = super()._pi_hours(obs)
        if active and self.OCC_GATE and self._occ_idx is not None:
            hour, _ = self._get_hour_and_day(obs)
            if hour >= self.morning_start_hour + 2 and float(obs[self._occ_idx]) < 0.5:
                return False
        return active

    def _zone_target(self, obs, lo, hi):
        centre = 0.5 * (lo + hi)
        if self.TARGET_SHIFT_K <= 0 or self._oat_any is None:
            return centre
        outside = _outside_kelvin(self, obs[self._oat_any])
        shift = min(self.TARGET_SHIFT_K, 0.4 * (hi - lo))
        if outside < centre:
            return centre - shift
        return centre if self.COLD_SHIFT_ONLY else centre + shift

    def _zone_band(self, obs, vav_id, low, high):
        lo, hi = super()._zone_band(obs, vav_id, low, high)
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            lo -= self.NIGHT_HEAT_DROP_K
        return lo, hi

    def _ahu_error(self, obs, ctrl, low, high):
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            low -= self.NIGHT_HEAT_DROP_K
        return super()._ahu_error(obs, ctrl, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if (self.WATER_MAX_PI and self._pi_active and self.water_temp_idx is not None
                and self._max_heat > 0):
            action["continuous_action"][self.water_temp_idx] = np.float32(1.0)
        if self.WATER_SCALE != 1.0 and self.water_temp_idx is not None:
            cont = action["continuous_action"]
            frac = (float(cont[self.water_temp_idx]) + 1.0) * 0.5 * self.WATER_SCALE
            cont[self.water_temp_idx] = np.float32(2.0 * frac - 1.0)
        return action


class TunedDemandSatPolicy(TunedPIPolicy, DemandSatPolicy):
    """dsat_bin with the TunedPIPolicy switches (all off by default)."""
    WARM_SAT = None  # dsat_bin did not use the cold-weather warm supply
    KEY = "tuned_dsat"


class AdaptiveColdPolicy(TunedDemandSatPolicy):
    """Pick the heating measures from a smoothed outside temperature.

    Screening showed each cold-weather measure helps one regime and hurts
    another: an early start and full-temperature water with less heating air
    pay off only in deep cold (mean -21 C), while aiming 0.5 K below the band
    centre pays off in moderate cold but hurts deep cold. The regime follows an
    exponential average of outside air temperature over about a day.
    """
    NIGHT_HEAT_DROP_K = 2.0
    PRESTART_H = 2
    NIGHT_COOL_CEILING = False
    EXTREME_BELOW_C = -15.0
    MILD_BELOW_C = 12.0
    EXTREME_PRESTART_H = 4
    EXTREME_WATER_MAX = True
    EXTREME_HEAT_DAMPER = 0.5
    MILD_SHIFT_K = 0.5
    EMA_STEPS = 288
    # Full hot-water temperature in PI hours whenever a zone heats, any weather.
    ALWAYS_WATER_MAX = False
    KEY = "adaptive_cold"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._oat_ema = None
        self._base_prestart = self.PRESTART_H

    def _update_regime(self, obs):
        if self._oat_any is not None:
            outside = _outside_kelvin(self, obs[self._oat_any]) - 273.15
            if self._oat_ema is None:
                self._oat_ema = outside
            else:
                self._oat_ema += (outside - self._oat_ema) / self.EMA_STEPS
        ema = self._oat_ema
        extreme = ema is not None and ema < self.EXTREME_BELOW_C
        mild = ema is not None and not extreme and ema < self.MILD_BELOW_C
        self.PRESTART_H = self.EXTREME_PRESTART_H if extreme else self._base_prestart
        self.WATER_MAX_PI = bool((extreme and self.EXTREME_WATER_MAX) or self.ALWAYS_WATER_MAX)
        self.HEAT_DAMPER_GAIN = self.EXTREME_HEAT_DAMPER if extreme else 1.0
        self.TARGET_SHIFT_K = self.MILD_SHIFT_K if mild else 0.0
        self.COLD_SHIFT_ONLY = True

    def _base_action(self, obs):
        self._update_regime(obs)
        return super()._base_action(obs)


# ---------------------------------------------------------------------------
# Additions for offline evaluation (not part of the reference notebook).
# ---------------------------------------------------------------------------

class RefPolicy(AdaptiveColdPolicy):
    """SubmissionPolicy of the reference notebook, minus the server base class."""
    KP = 0.1
    KI = 0.005
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = False
    EXTREME_WATER_MAX = False
    EXTREME_HEAT_DAMPER = 1.0
    KEY = "ref"


class NudgedRefPolicy(RefPolicy):
    """RefPolicy with a negligible offset on every AHU static-pressure command.

    1e-5 in normalized units is about 0.01 Pa: physically nothing, but enough
    to flip a later switching decision. The spread of these runs is the noise
    floor a real change has to clear.
    """
    NUDGE = 0.0

    def get_action(self, obs):
        action = super().get_action(obs)
        cont = action["continuous_action"]
        for ctrl in self.ahu_controllers:
            j = ctrl.get("pressure_idx")
            if j is not None:
                cont[j] = np.float32(np.clip(cont[j] + self.NUDGE, -1.0, 1.0 - 1e-6))
        return action


def make_policy(variant, building_meta_data, action_spec):
    """'ref', 'null_<k>' (pressure nudge k * 1e-5) or 'bang_bang'."""
    if variant == "ref":
        return RefPolicy(building_meta_data, action_spec)
    if variant.startswith("null_"):
        k = int(variant.split("_")[1])
        cls = type(f"Null{k}Policy", (NudgedRefPolicy,), {"NUDGE": k * 1e-5})
        return cls(building_meta_data, action_spec)
    if variant == "bang_bang":
        from smart_buildings.baselines.bang_bang import BangBangPolicy
        return BangBangPolicy(building_meta_data, action_spec)
    raise ValueError(f"unknown variant: {variant}")


## Bộ chạy

Mỗi kịch bản chạy trong một tiến trình riêng, cùng vòng lặp với `evaluate.py` của starter kit.

In [ ]:
%%writefile sb_eval_worker.py
"""Run one policy variant on one training scenario and save its summary.

Runs in a separate process. Mirrors the starter kit's evaluate.py loop
(SimulationTracker, occupancy constant 125, validate_action) so returns are
comparable with baseline_returns.json.
"""
import json
import os
import sys
import time
from pathlib import Path

import numpy as np

STEPS_PER_DAY = 288


def _data_root() -> Path:
    # Absolute paths: the worker chdirs into the data root, so '' on sys.path
    # would no longer point at the folder holding ref_policy.py.
    for p in (os.environ.get("SB_WORK"), os.environ["SB_DATA_ROOT"]):
        if p and p not in sys.path:
            sys.path.insert(0, p)
    return Path(os.environ["SB_DATA_ROOT"])


def result_path(out_dir, variant, scenario) -> Path:
    return Path(out_dir) / variant / (scenario.replace("/", "__") + ".json")


def run_scenario(scenario: str, variant: str, n_steps: int, out_dir: str, save_steps: bool = True) -> dict:
    root = _data_root()
    # config.gin files reference floor plans by relative path.
    os.chdir(root)
    from smart_buildings.evaluate import (
        create_building_meta_data, load_hybrid_environment, validate_action)
    from smart_buildings.smart_control.utils.scenario_generator import SimulationTracker
    import ref_policy

    sdir = root / "smart_buildings" / "training_scenarios" / scenario
    t_start = time.perf_counter()
    env = load_hybrid_environment(sdir / "config.gin")
    env.metrics_path = None  # Kaggle input is read-only
    env._occupancy_normalization_constant = 125.0  # as in evaluate.py
    tracker = SimulationTracker(env, vmin=280, vmax=310, capture_snapshots=False)
    meta = create_building_meta_data(env, sdir)
    spec = env.action_spec()
    policy = ref_policy.make_policy(variant, meta, spec)

    tracker.reset()
    obs = tracker.last_obs
    t_policy = t_env = 0.0
    for _ in range(n_steps):
        t0 = time.perf_counter()
        action = validate_action(policy.get_action(obs), spec)
        t1 = time.perf_counter()
        obs, _, terminated, truncated, _ = tracker.step(action)
        t2 = time.perf_counter()
        t_policy += t1 - t0
        t_env += t2 - t1
        if terminated or truncated:
            break

    rewards = np.asarray(tracker.rewards, dtype=float)
    rc = tracker.reward_components
    regret = np.array([r["normalized_productivity_regret"] for r in rc], dtype=float)
    energy = np.array([r["normalized_energy_cost"] for r in rc], dtype=float)
    carbon = np.array([r["normalized_carbon_emission"] for r in rc], dtype=float)
    occ = np.asarray(tracker.occupancy, dtype=float)
    wp, we, wc = tracker.productivity_weight, tracker.energy_weight, tracker.carbon_weight
    comfort_s, energy_s, carbon_s = wp * np.abs(regret), we * energy, wc * carbon
    run = np.array([d["ahu_run_cmds"] for d in tracker.ahu_log], dtype=int)
    outside = np.array([d["outside_air"] for d in tracker.ahu_log], dtype=float)
    temps = np.asarray(tracker.zone_temps, dtype=float)
    T = len(rewards)
    occupied = occ > 0

    summary = {
        "scenario": scenario,
        "building": scenario.split("/")[0],
        "weather": scenario.split("/")[1],
        "station": scenario.split("/")[1].rsplit("_", 1)[0],
        "variant": variant,
        "steps": T,
        "total_reward": float(rewards.sum()),
        "score": float(abs(rewards.mean())),
        "comfort_loss": float(comfort_s.mean()),
        "energy_loss": float(energy_s.mean()),
        "carbon_loss": float(carbon_s.mean()),
        # Sum of the three parts minus the score; ~0 when the decomposition is exact.
        "decomposition_gap": float(comfort_s.mean() + energy_s.mean() + carbon_s.mean() - abs(rewards.mean())),
        "occupied_steps": int(occupied.sum()),
        "comfort_violation_frac": float((regret[occupied] != 0).mean()) if occupied.any() else 0.0,
        "worst_step_comfort": float(comfort_s.max()) if T else 0.0,
        "zone_temp_min_occ_C": float(temps[occupied].min() - 273.15) if occupied.any() else None,
        "zone_temp_max_occ_C": float(temps[occupied].max() - 273.15) if occupied.any() else None,
        "outside_mean_C": float(outside.mean() - 273.15) if T else None,
        "n_zones": int(temps.shape[1]) if temps.ndim == 2 else None,
        "n_ahus": int(run.shape[1]) if run.ndim == 2 else None,
        "ahu_switches_per_ahu_day": float(np.abs(np.diff(run, axis=0)).sum(0).mean() / max(T / STEPS_PER_DAY, 1e-9)) if run.ndim == 2 and T > 1 else None,
        "ahu_on_frac": float(run.mean()) if run.size else None,
        "seconds_policy": t_policy,
        "seconds_env": t_env,
        "seconds_total": time.perf_counter() - t_start,
    }

    path = result_path(out_dir, variant, scenario)
    path.parent.mkdir(parents=True, exist_ok=True)
    if save_steps:
        import pandas as pd
        pd.DataFrame({
            "reward": rewards, "comfort": comfort_s, "energy": energy_s, "carbon": carbon_s,
            "occupancy": occ, "outside_C": outside - 273.15, "ahus_on": run.sum(1) if run.ndim == 2 else 0,
            "zone_temp_min_C": temps.min(1) - 273.15, "zone_temp_max_C": temps.max(1) - 273.15,
        }).to_csv(path.with_suffix(".steps.csv.gz"), index=False, float_format="%.6g")
    path.write_text(json.dumps(summary, indent=2))
    return summary


In [ ]:
import importlib
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd

import sb_eval_worker
importlib.reload(sb_eval_worker)
from sb_eval_worker import result_path, run_scenario

SCEN_ROOT = DATA_ROOT / "smart_buildings" / "training_scenarios"
SCENARIOS = sorted(f"{d.parent.name}/{d.name}" for d in SCEN_ROOT.glob("*/*") if (d / "config.gin").exists())
if ONLY:
    SCENARIOS = [s for s in SCENARIOS if s in ONLY]
# Bigger floor plans run slower: start them first so they do not finish last.
SIZE = {s: np.load(SCEN_ROOT / s / "floor_plan.npy", mmap_mode="r").size for s in SCENARIOS}
print(len(SCENARIOS), "scenarios")


def run_batch(jobs, workers=None, out=OUT, save_steps=True):
    """jobs: (scenario, variant, steps). Finished jobs are skipped, so re-running resumes."""
    todo = [j for j in jobs
            if not (result_path(out, j[1], j[0]).exists()
                    and json.loads(result_path(out, j[1], j[0]).read_text()).get("steps") == j[2])]
    todo.sort(key=lambda j: -SIZE.get(j[0], 0))
    workers = workers or WORKERS
    print(f"{len(jobs) - len(todo)} done already, {len(todo)} to run on {workers} workers")
    if not todo:
        return
    kw = {"max_workers": workers, "mp_context": mp.get_context("spawn")}
    if sys.version_info >= (3, 11):
        kw["max_tasks_per_child"] = 1
    t0 = time.time()
    with ProcessPoolExecutor(**kw) as ex:
        futs = {ex.submit(run_scenario, sc, var, steps, str(out), save_steps): (sc, var) for sc, var, steps in todo}
        for i, f in enumerate(as_completed(futs), 1):
            sc, var = futs[f]
            try:
                s = f.result()
                print(f"[{i:2d}/{len(todo)}] {time.time() - t0:6.0f}s  {var:9s} {sc:30s} "
                      f"score {s['score']:.4f}  ({s['seconds_total'] / 60:.1f} min)", flush=True)
            except Exception as e:
                print(f"[{i:2d}/{len(todo)}] FAILED {var} {sc}: {e!r}", flush=True)


def load(variant, out=OUT):
    rows = [json.loads(p.read_text()) for p in sorted((out / variant).glob("*.json"))]
    df = pd.DataFrame(rows)
    return df[df.steps == N_STEPS].reset_index(drop=True) if len(df) else df

## Chạy 30 kịch bản

In [ ]:
run_batch([(s, "ref", N_STEPS) for s in SCENARIOS])

## Kết quả

Xếp theo phần đóng góp vào tổng điểm (trung bình thô, nên kịch bản tệ nhất quyết định nhiều nhất).

In [ ]:
ref = load("ref").sort_values("score", ascending=False).reset_index(drop=True)
ref["share_pct"] = 100 * ref.score / ref.score.sum()
ref["cum_share_pct"] = ref.share_pct.cumsum()

bfile = DATA_ROOT / "baseline_returns.json"
HAS_BASE = bfile.exists() and N_STEPS == 8640   # baseline returns are full-episode only
if HAS_BASE:
    base = json.loads(bfile.read_text())
    ref["pid_water"] = ref.scenario.map(lambda s: abs(base[s]["pid_water"]) / 8640 if s in base else np.nan)
    ref["vs_pid_water_pct"] = 100 * (ref.score - ref.pid_water) / ref.pid_water

print(f"{len(ref)} scenarios, {N_STEPS} steps each")
print(f"score   {ref.score.mean():.5f}")
print(f"  comfort {ref.comfort_loss.mean():.5f}   energy {ref.energy_loss.mean():.5f}   carbon {ref.carbon_loss.mean():.5f}")
if HAS_BASE:
    print(f"pid_water on the same scenarios: {ref.pid_water.mean():.5f}")
print(f"max |decomposition gap|: {ref.decomposition_gap.abs().max():.1e}  (should be ~0)")

cols = ["scenario", "score", "share_pct", "cum_share_pct", "comfort_loss", "energy_loss", "carbon_loss",
        "comfort_violation_frac", "zone_temp_min_occ_C", "zone_temp_max_occ_C", "ahu_switches_per_ahu_day"]
cols += ["pid_water", "vs_pid_water_pct"] if HAS_BASE else []
display(ref[cols].round(4))

In [ ]:
by_station = (ref.groupby("station")
              .agg(score=("score", "mean"), comfort=("comfort_loss", "mean"),
                   energy=("energy_loss", "mean"), carbon=("carbon_loss", "mean"),
                   share_pct=("share_pct", "sum"), scenarios=("scenario", "count"))
              .sort_values("share_pct", ascending=False))
display(by_station.round(4))

In [ ]:
import matplotlib.pyplot as plt

C = {"comfort": "#2f6fb3", "energy": "#d9551c", "carbon": "#677689"}
fig = plt.figure(figsize=(14, max(6, 0.28 * len(ref) + 1)))
gs = fig.add_gridspec(2, 2, width_ratios=[1.35, 1], height_ratios=[1.1, 1])

ax = fig.add_subplot(gs[:, 0])
y = np.arange(len(ref))[::-1]
left = np.zeros(len(ref))
for k, col in (("comfort", "comfort_loss"), ("energy", "energy_loss"), ("carbon", "carbon_loss")):
    ax.barh(y, ref[col], left=left, color=C[k], label=k)
    left += ref[col].to_numpy()
ax.set_yticks(y, ref.scenario, fontsize=8)
ax.set_xlabel("điểm (thấp hơn = tốt hơn)")
ax.set_title("Điểm từng kịch bản, tách theo thành phần")
ax.legend(loc="lower right", frameon=False)
for yi, (v, sh) in zip(y, zip(ref.score, ref.share_pct)):
    ax.text(v, yi, f"  {sh:.0f}%", va="center", fontsize=7, color="#555")

ax = fig.add_subplot(gs[0, 1])
piv = ref.pivot(index="building", columns="station", values="score")
logv = np.log10(piv.to_numpy(dtype=float))
im = ax.imshow(logv, cmap="Oranges", aspect="auto")
ax.set_xticks(range(piv.shape[1]), piv.columns, fontsize=8)
ax.set_yticks(range(piv.shape[0]), [b.replace("building_train_", "Tòa ") for b in piv.index], fontsize=8)
lo, hi = np.nanmin(logv), np.nanmax(logv)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.iat[i, j]
        if np.isfinite(v):
            dark = hi > lo and (logv[i, j] - lo) / (hi - lo) > 0.6
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7, color="white" if dark else "black")
ax.set_title("Điểm theo tòa nhà × khí hậu (đậm = tệ)")

ax = fig.add_subplot(gs[1, 1])
ax.bar(by_station.index, by_station.share_pct, color="#677689")
ax.axhline(100 / len(by_station), color="#d9551c", ls="--", lw=1, label="nếu chia đều")
ax.set_ylabel("% tổng điểm")
ax.set_title("Phần đóng góp theo khí hậu")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "ref_scores.png", dpi=130)
plt.show()

## Nhóm null: đo nhiễu

Chạy lại các kịch bản đóng góp nhiều nhất với lệnh áp suất lệch ~0,01 Pa. Về vật lý không có gì thay đổi,
nên độ phân tán giữa các lần chạy là mức nhiễu. Một thay đổi thật phải vượt khoảng 2 độ lệch chuẩn.

In [ ]:
noise = None
if RUN_NULL and len(ref):
    top = ref.scenario.head(NULL_TOP).tolist()
    run_batch([(s, f"null_{k}", N_STEPS) for s in top for k in range(1, NULL_K + 1)])
    nul = pd.concat([load(f"null_{k}") for k in range(1, NULL_K + 1)], ignore_index=True)
    grp = pd.concat([ref.loc[ref.scenario.isin(top), ["scenario", "score"]],
                     nul.loc[nul.scenario.isin(top), ["scenario", "score"]]])
    noise = grp.groupby("scenario").score.agg(["mean", "std", "min", "max", "count"])
    noise["sd_pct"] = 100 * noise["std"] / noise["mean"]
    noise["range_pct"] = 100 * (noise["max"] - noise["min"]) / noise["mean"]
    noise = noise.loc[top]
    display(noise.round(5))
    print(f"median sd: {noise.sd_pct.median():.1f}% of the score, max range: {noise.range_pct.max():.1f}%")


def z_scores(candidate):
    """candidate: {scenario: score}. z < -2 = real improvement, z > 2 = real regression."""
    return {s: (v - noise.at[s, "mean"]) / noise.at[s, "std"] for s, v in candidate.items() if s in noise.index}

## Thời gian chạy so với bang-bang

Luật: mỗi rollout ≤ 10× thời gian của bang-bang. Đo trên tòa nhà lớn nhất, chạy tuần tự để hai bên không tranh CPU.

In [ ]:
timing = None
if RUN_TIMING and SCENARIOS:
    big = max(SCENARIOS, key=SIZE.get)
    tdir = OUT / "timing"
    for var in ("bang_bang", "ref"):
        run_batch([(big, var, TIMING_STEPS)], workers=1, out=tdir, save_steps=False)
    rows = [json.loads(result_path(tdir, v, big).read_text()) for v in ("bang_bang", "ref")]
    timing = pd.DataFrame(rows).set_index("variant")[["steps", "seconds_policy", "seconds_env"]]
    timing["seconds_rollout"] = timing.seconds_policy + timing.seconds_env
    timing["ms_per_step"] = 1000 * timing.seconds_rollout / timing.steps
    display(timing.round(3))
    ratio = timing.at["ref", "seconds_rollout"] / timing.at["bang_bang", "seconds_rollout"]
    print(f"{big}: ref / bang_bang = {ratio:.2f}x  (limit 10x)")

## Lưu kết quả

In [ ]:
ref.to_csv(OUT / "scores_ref.csv", index=False)
by_station.to_csv(OUT / "scores_by_station.csv")
if noise is not None:
    noise.to_csv(OUT / "noise.csv")
summary = {
    "policy": "ref (Smart Buildings Probe DS K07, public LB 0.01582)",
    "steps": N_STEPS,
    "scenarios": int(len(ref)),
    "score": float(ref.score.mean()),
    "comfort_loss": float(ref.comfort_loss.mean()),
    "energy_loss": float(ref.energy_loss.mean()),
    "carbon_loss": float(ref.carbon_loss.mean()),
    "pid_water_same_scenarios": float(ref.pid_water.mean()) if HAS_BASE else None,
    "top5_share_pct": float(ref.share_pct.head(5).sum()),
    "noise_median_sd_pct": float(noise.sd_pct.median()) if noise is not None else None,
    "runtime_ratio_vs_bang_bang": float(ratio) if timing is not None else None,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Files:", sorted(p.name for p in OUT.iterdir()))